# Cross-Version External Validation: v3.0.0 vs v2.0.1

This notebook performs bidirectional cross-version external validation between
Bridge2AI Voice v3.0.0 and v2.0.1 datasets.

**Direction 1:** Train on v3.0.0 (5-fold models) -> Test on v2.0.1 (ensemble)

**Direction 2:** Train on v2.0.1 (5-fold models) -> Test on v3.0.0 (ensemble)

### Key differences between versions
- **Participant IDs:** v2.0.1 uses 8-character hex IDs; v3.0.0 uses 6-digit numeric IDs -- no participant overlap is possible
- **Spectrogram resolution:** v2.0.1 has 201 mel bins; v3.0.0 has 60 mel bins -- both are resized to 128 for AST
- **Phenotype format:** v2.0.1 uses a flat `phenotype.tsv` with `parkinsons` column (Checked/Unchecked); v3.0.0 uses hierarchical `diagnosis/parkinsons_disease.tsv` + `control.tsv`
- **Task names:** Different naming conventions between versions
- **Recording conditions:** Potentially different sites, equipment, and recording protocols

This cross-version validation tests whether PD screening models generalize across
dataset versions with different recording conditions and preprocessing.

In [ ]:
#1 - Imports and configuration
import os
import json
import copy
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm import tqdm
from scipy.ndimage import zoom

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig

from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score,
    precision_score, recall_score, confusion_matrix, accuracy_score
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# --- Paths ---
PROJECT_ROOT = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast')
RESULTS_DIR = PROJECT_ROOT / 'results' / 'v3'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# v3.0.0 paths
V3_ROOT = Path('/data0/b2ai-voice/3.0.0')
V3_SPEC = V3_ROOT / 'features' / 'torchaudio_mel_spectrogram.parquet'
V3_PD_PHEN = V3_ROOT / 'phenotype' / 'diagnosis' / 'parkinsons_disease.tsv'
V3_CTRL_PHEN = V3_ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'

# v2.0.1 paths
V2_ROOT = Path('/data0/b2ai-voice/2.0.0')
V2_SPEC = V2_ROOT / 'spectrogram.parquet'
V2_PHEN = V2_ROOT / 'phenotype.tsv'

# Model paths
V3_MODEL_PATTERN = str(RESULTS_DIR / 'ast_pd_v3_fold{}.pt')
V3_CV_RESULTS = RESULTS_DIR / 'ast_pd_v3_cv_results.npz'
V2_MODEL_PATTERN = str(PROJECT_ROOT / 'ast_pd_selected_fold{}.pt')
V2_CV_RESULTS = PROJECT_ROOT / 'ast_pd_selected_cv_results.npz'

# Output
OUTPUT_JSON = RESULTS_DIR / 'cross_version_validation.json'

print(f'v3 spec exists: {V3_SPEC.exists()}')
print(f'v2 spec exists: {V2_SPEC.exists()}')
print(f'v3 CV results exist: {V3_CV_RESULTS.exists()}')
print(f'v2 CV results exist: {V2_CV_RESULTS.exists()}')

In [ ]:
#2 - Model definition (must match training architecture exactly)
class ASTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained_tag='MIT/ast-finetuned-audioset-10-10-0.4593'):
        super().__init__()
        self.ast = ASTModel.from_pretrained(pretrained_tag)
        hidden = self.ast.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, pixel_values):
        out = self.ast(pixel_values).last_hidden_state[:, 0, :]
        return self.classifier(out)

class SpectrogramDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, participants):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        x = self.X[idx]
        x_3ch = x.unsqueeze(0).expand(3, -1, -1)
        return {'pixel_values': x_3ch.permute(0, 2, 1), 'labels': self.y[idx], 'participant': self.participants[idx]}

def process_spectrogram(spec_raw, target_freq=128, target_time=1024):
    spec = np.stack(spec_raw).astype(np.float32)
    n_mel, time_len = spec.shape
    if time_len < target_time:
        spec = np.pad(spec, ((0, 0), (0, target_time - time_len)), mode='reflect')
    elif time_len > target_time:
        start = (time_len - target_time) // 2
        spec = spec[:, start:start + target_time]
    freq_ratio = target_freq / n_mel
    spec = zoom(spec, (freq_ratio, 1.0), order=1)
    return spec

print('Model and helpers defined.')

In [ ]:
#3 - Load and preprocess v2.0.1 data (PD cohort)
MIN_TIME_FRAMES = 100

# v2.0.1 phenotype
v2_pheno = pd.read_csv(V2_PHEN, sep='\t')
v2_pheno['participant_id'] = v2_pheno['participant_id'].astype(str)
v2_pd_ids = set(v2_pheno[v2_pheno['parkinsons'] == 'Checked']['participant_id'])
v2_ctrl_ids = set(v2_pheno[v2_pheno['parkinsons'] != 'Checked']['participant_id'])
# Also exclude anyone who is PD from controls
v2_ctrl_ids = v2_ctrl_ids - v2_pd_ids
print(f'v2.0.1 PD: {len(v2_pd_ids)}, Ctrl: {len(v2_ctrl_ids)}')

# v2.0.1 spectrograms - load selected tasks to match v2 training
# Use same tasks as original v2 PD experiments
V2_SELECTED_TASKS = [
    'Cinderella-Story', 'Productive-Vocabulary-1', 'Productive-Vocabulary-2',
    'Productive-Vocabulary-3', 'Productive-Vocabulary-4', 'Productive-Vocabulary-5',
    'Productive-Vocabulary-6', 'Word-Color-Stroop', 'Random-Item-Generation'
]

pf2 = pq.ParquetFile(V2_SPEC)
parts2 = []
for i in range(pf2.num_row_groups):
    parts2.append(pf2.read_row_group(i, columns=['participant_id','session_id','task_name','spectrogram','n_frames']).to_pandas())
v2_spec = pd.concat(parts2, ignore_index=True)
v2_spec['participant_id'] = v2_spec['participant_id'].astype(str)

v2_spec['label'] = np.nan
v2_spec.loc[v2_spec['participant_id'].isin(v2_pd_ids), 'label'] = 1
v2_spec.loc[v2_spec['participant_id'].isin(v2_ctrl_ids), 'label'] = 0
v2_data = v2_spec.dropna(subset=['label']).copy()
v2_data['label'] = v2_data['label'].astype(int)
v2_data = v2_data[(v2_data['task_name'].isin(V2_SELECTED_TASKS)) & (v2_data['n_frames'] >= MIN_TIME_FRAMES)].copy()

print(f'v2 data: {len(v2_data)} recordings, {v2_data["participant_id"].nunique()} participants')
print(f'  PD: {(v2_data["label"]==1).sum()} recs, Ctrl: {(v2_data["label"]==0).sum()} recs')

# Process v2 spectrograms to (128, 1024)
X_v2_list = []
for idx, row in tqdm(v2_data.iterrows(), total=len(v2_data), desc='Processing v2 specs'):
    X_v2_list.append(process_spectrogram(row['spectrogram'], 128, 1024))
X_v2 = np.stack(X_v2_list)
y_v2 = v2_data['label'].values
p_v2 = v2_data['participant_id'].values
print(f'v2 shape: {X_v2.shape}')

In [ ]:
#4 - Load and preprocess v3.0.0 data (PD cohort, selected tasks)
SELECTED_TASKS = [
    'prolonged-vowel', 'glides-high-to-low', 'glides-low-to-high',
    'diadochokinesis-pataka', 'rainbow-passage', 'picture-description',
    'story-recall', 'maximum-phonation-time-1',
]

# v3.0.0 phenotype
v3_pd_pheno = pd.read_csv(V3_PD_PHEN, sep='\t')
v3_ctrl_pheno = pd.read_csv(V3_CTRL_PHEN, sep='\t')
v3_pd_ids = set(v3_pd_pheno['participant_id'].astype(str).str.zfill(6))
v3_ctrl_ids = set(v3_ctrl_pheno['participant_id'].astype(str).str.zfill(6)) - v3_pd_ids
print(f'v3.0.0 PD: {len(v3_pd_ids)}, Ctrl: {len(v3_ctrl_ids)}')

# v3.0.0 spectrograms
pf3 = pq.ParquetFile(V3_SPEC)
parts3 = []
for i in range(pf3.num_row_groups):
    parts3.append(pf3.read_row_group(i, columns=['participant_id','session_id','task_name','mel_spectrogram','n_frames']).to_pandas())
v3_spec = pd.concat(parts3, ignore_index=True)
v3_spec['participant_id'] = v3_spec['participant_id'].astype(str).str.zfill(6)

v3_spec['label'] = np.nan
v3_spec.loc[v3_spec['participant_id'].isin(v3_pd_ids), 'label'] = 1
v3_spec.loc[v3_spec['participant_id'].isin(v3_ctrl_ids), 'label'] = 0
v3_data = v3_spec.dropna(subset=['label']).copy()
v3_data['label'] = v3_data['label'].astype(int)
v3_data = v3_data[(v3_data['task_name'].isin(SELECTED_TASKS)) & (v3_data['n_frames'] >= MIN_TIME_FRAMES)].copy()

print(f'v3 data: {len(v3_data)} recordings, {v3_data["participant_id"].nunique()} participants')
print(f'  PD: {(v3_data["label"]==1).sum()} recs, Ctrl: {(v3_data["label"]==0).sum()} recs')

# Process v3 spectrograms to (128, 1024)
X_v3_list = []
for idx, row in tqdm(v3_data.iterrows(), total=len(v3_data), desc='Processing v3 specs'):
    X_v3_list.append(process_spectrogram(row['mel_spectrogram'], 128, 1024))
X_v3 = np.stack(X_v3_list)
y_v3 = v3_data['label'].values
p_v3 = v3_data['participant_id'].values
print(f'v3 shape: {X_v3.shape}')

In [ ]:
#5 - Evaluation helper
def evaluate_ensemble(models, loader, device):
    """Average predictions from multiple models, aggregate to participant level."""
    all_probs_per_model = [[] for _ in models]
    all_labels, all_parts = [], []
    
    with torch.no_grad():
        for batch in loader:
            pv = batch['pixel_values'].to(device)
            for i, model in enumerate(models):
                model.eval()
                outputs = model(pv)
                probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
                all_probs_per_model[i].extend(probs)
            all_labels.extend(batch['labels'].numpy())
            all_parts.extend(batch['participant'])
    
    # Average across models (ensemble)
    all_probs_per_model = [np.array(p) for p in all_probs_per_model]
    ensemble_probs = np.mean(all_probs_per_model, axis=0)
    all_labels = np.array(all_labels)
    all_parts = np.array(all_parts)
    
    # Aggregate to participant level
    unique_parts = np.unique(all_parts)
    part_probs = np.array([ensemble_probs[all_parts == p].mean() for p in unique_parts])
    part_labels = np.array([all_labels[all_parts == p][0] for p in unique_parts])
    
    return part_probs, part_labels, unique_parts

print('Evaluation helper defined.')

In [ ]:
#6 - Direction 1: Train v3.0.0 -> Test v2.0.1
# Load v3 fold models and CV results for normalization stats
v3_cv = np.load(str(V3_CV_RESULTS), allow_pickle=True)
print(f'v3 OOF AUC: {float(v3_cv["oof_auc"]):.4f}')

# Load 5 v3 fold models
v3_models = []
for fold in range(5):
    model = ASTClassifier(num_classes=2).to(device)
    model.load_state_dict(torch.load(V3_MODEL_PATTERN.format(fold), map_location=device))
    model.eval()
    v3_models.append(model)
    print(f'  Loaded v3 fold {fold}')

# Normalize v2 data using v3 training stats (global mean/std from full v3 training data)
# Use overall v3 mean/std as approximation since we don't have per-fold stats saved separately
v3_mean, v3_std = X_v3.mean(), X_v3.std()
X_v2_norm = (X_v2 - v3_mean) / (v3_std + 1e-8)

v2_ds = SpectrogramDataset(X_v2_norm, y_v2, p_v2)
v2_loader = torch.utils.data.DataLoader(v2_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

pp_v2, pl_v2, pids_v2 = evaluate_ensemble(v3_models, v2_loader, device)
auc_v3_to_v2 = roc_auc_score(pl_v2, pp_v2)
fpr, tpr, thr = roc_curve(pl_v2, pp_v2)
ot = thr[np.argmax(tpr-fpr)]
preds_v2 = (pp_v2 >= ot).astype(int)

print(f'\n=== Direction 1: v3.0.0 -> v2.0.1 ===')
print(f'Test participants: {len(pids_v2)} (PD: {int(pl_v2.sum())}, Ctrl: {int((1-pl_v2).sum())})')
print(f'AUC: {auc_v3_to_v2:.4f}')
print(f'F1:  {f1_score(pl_v2, preds_v2, zero_division=0):.4f}')
print(f'Recall: {recall_score(pl_v2, preds_v2, zero_division=0):.4f}')
print(f'Precision: {precision_score(pl_v2, preds_v2, zero_division=0):.4f}')

# Cleanup v3 models
del v3_models; torch.cuda.empty_cache()

In [ ]:
#7 - Direction 2: Train v2.0.1 -> Test v3.0.0
# Check if v2 fold models exist
v2_models_exist = all(Path(V2_MODEL_PATTERN.format(f)).exists() for f in range(5))
print(f'v2 fold models exist: {v2_models_exist}')

if v2_models_exist:
    # Load 5 v2 fold models
    v2_models = []
    for fold in range(5):
        model = ASTClassifier(num_classes=2).to(device)
        model.load_state_dict(torch.load(V2_MODEL_PATTERN.format(fold), map_location=device))
        model.eval()
        v2_models.append(model)
        print(f'  Loaded v2 fold {fold}')

    # Normalize v3 data using v2 training stats
    v2_mean, v2_std = X_v2.mean(), X_v2.std()
    X_v3_norm = (X_v3 - v2_mean) / (v2_std + 1e-8)

    v3_ds = SpectrogramDataset(X_v3_norm, y_v3, p_v3)
    v3_loader = torch.utils.data.DataLoader(v3_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    pp_v3, pl_v3, pids_v3 = evaluate_ensemble(v2_models, v3_loader, device)
    auc_v2_to_v3 = roc_auc_score(pl_v3, pp_v3)
    fpr, tpr, thr = roc_curve(pl_v3, pp_v3)
    ot = thr[np.argmax(tpr-fpr)]
    preds_v3 = (pp_v3 >= ot).astype(int)

    print(f'\n=== Direction 2: v2.0.1 -> v3.0.0 ===')
    print(f'Test participants: {len(pids_v3)} (PD: {int(pl_v3.sum())}, Ctrl: {int((1-pl_v3).sum())})')
    print(f'AUC: {auc_v2_to_v3:.4f}')
    print(f'F1:  {f1_score(pl_v3, preds_v3, zero_division=0):.4f}')
    print(f'Recall: {recall_score(pl_v3, preds_v3, zero_division=0):.4f}')
    print(f'Precision: {precision_score(pl_v3, preds_v3, zero_division=0):.4f}')
    
    del v2_models; torch.cuda.empty_cache()
else:
    print('v2 fold models not found — skipping Direction 2.')
    print('Run the v2 PD experiment first to generate fold models.')
    auc_v2_to_v3 = None

In [ ]:
#8 - Summary and save results
results = {
    'v3_to_v2': {
        'train_version': 'v3.0.0',
        'test_version': 'v2.0.1',
        'train_n': int(len(np.unique(p_v3))),
        'test_n': int(len(pids_v2)),
        'auc': float(auc_v3_to_v2),
        'f1': float(f1_score(pl_v2, preds_v2, zero_division=0)),
        'recall': float(recall_score(pl_v2, preds_v2, zero_division=0)),
        'precision': float(precision_score(pl_v2, preds_v2, zero_division=0)),
    }
}

if auc_v2_to_v3 is not None:
    results['v2_to_v3'] = {
        'train_version': 'v2.0.1',
        'test_version': 'v3.0.0',
        'train_n': int(len(np.unique(p_v2))),
        'test_n': int(len(pids_v3)),
        'auc': float(auc_v2_to_v3),
        'f1': float(f1_score(pl_v3, preds_v3, zero_division=0)),
        'recall': float(recall_score(pl_v3, preds_v3, zero_division=0)),
        'precision': float(precision_score(pl_v3, preds_v3, zero_division=0)),
    }

with open(str(OUTPUT_JSON), 'w') as f:
    json.dump(results, f, indent=2)

print(f'\n=== Cross-Version Validation Summary ===')
print(f'v3.0.0 -> v2.0.1: AUC = {auc_v3_to_v2:.4f}')
if auc_v2_to_v3 is not None:
    print(f'v2.0.1 -> v3.0.0: AUC = {auc_v2_to_v3:.4f}')
print(f'\nSaved: {OUTPUT_JSON}')